# Task 3: Cuisine Classification

Develop a machine learning model to classify restaurants based on their cuisines using the restaurant dataset.

## 1. Data Loading and Exploration

In [1]:
import pandas as pd
df=pd.read_csv("Dataset.csv")
df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


### 1.1 Inspecting Cuisine Data

In [2]:
df["Cuisines"].head(20)

0             French, Japanese, Desserts
1                               Japanese
2       Seafood, Asian, Filipino, Indian
3                        Japanese, Sushi
4                       Japanese, Korean
5                                Chinese
6                        Asian, European
7     Seafood, Filipino, Asian, European
8                European, Asian, Indian
9                               Filipino
10                     Filipino, Mexican
11         American, Ice Cream, Desserts
12                     Filipino, Mexican
13                              Filipino
14                                Korean
15     Cafe, American, Italian, Filipino
16                        Italian, Pizza
17                              Filipino
18                Cafe, Korean, Desserts
19       Cafe, Bakery, American, Italian
Name: Cuisines, dtype: object

### 1.2 Creating the Primary Cuisine Target

The Cuisines column contains multiple cuisine names for some restaurants.
For this classification experiment, the first listed cuisine is treated as the primary cuisine label.

In [3]:
df["Cuisine_Target"] = df["Cuisines"].str.split(",").str[0].str.strip()

df["Cuisine_Target"].head(20)

0       French
1     Japanese
2      Seafood
3     Japanese
4     Japanese
5      Chinese
6        Asian
7      Seafood
8     European
9     Filipino
10    Filipino
11    American
12    Filipino
13    Filipino
14      Korean
15        Cafe
16     Italian
17    Filipino
18        Cafe
19        Cafe
Name: Cuisine_Target, dtype: object

### 1.3 Cuisine Class Distribution

The distribution of the cuisine classes is examined to understand the number of restaurants belonging to each primary cuisine category.

In [4]:
df["Cuisine_Target"].value_counts()

Cuisine_Target
North Indian    2992
Chinese          855
Fast Food        672
Bakery           621
Cafe             617
                ... 
Tex-Mex            1
Australian         1
Irish              1
Persian            1
Pub Food           1
Name: count, Length: 119, dtype: int64

### 1.4 Checking Rare Cuisine Classes

The cuisine class distribution shows that several cuisine categories contain very few restaurant records. These rare classes are examined before model training because insufficient samples can make reliable classification difficult.

In [5]:
class_counts = df["Cuisine_Target"].value_counts()

class_counts[class_counts < 10]

Cuisine_Target
Korean            9
Coffee and Tea    9
Bar Food          9
Pakistani         8
Latin American    7
                 ..
Tex-Mex           1
Australian        1
Irish             1
Persian           1
Pub Food          1
Name: count, Length: 71, dtype: int64

### 1.5 Selecting a Minimum Cuisine Frequency

Different minimum frequency values are compared to determine a suitable number of restaurant records required for a cuisine category to be included in the classification dataset.

In [6]:
class_counts = df["Cuisine_Target"].value_counts()

for threshold in [5, 10, 20, 30, 50]:
    print(
        f"Minimum frequency {threshold}: "
        f"{(class_counts >= threshold).sum()} cuisines, "
        f"{class_counts[class_counts >= threshold].sum()} restaurants"
    )

Minimum frequency 5: 71 cuisines, 9444 restaurants
Minimum frequency 10: 48 cuisines, 9298 restaurants
Minimum frequency 20: 33 cuisines, 9097 restaurants
Minimum frequency 30: 27 cuisines, 8947 restaurants
Minimum frequency 50: 26 cuisines, 8898 restaurants


### 1.6 Filtering Rare Cuisine Classes

Cuisine categories with fewer than 10 restaurant records are excluded from the classification dataset.
This reduces extremely rare classes while retaining the majority of the available restaurant records.

In [8]:
min_frequency = 10

class_counts = df["Cuisine_Target"].value_counts()

valid_cuisines = class_counts[class_counts >= min_frequency].index

df_classification = df[
    df["Cuisine_Target"].isin(valid_cuisines)
].copy()

df_classification["Cuisine_Target"].value_counts()

Cuisine_Target
North Indian     2992
Chinese           855
Fast Food         672
Bakery            621
Cafe              617
American          278
South Indian      262
Mithai            246
Street Food       236
Continental       235
Italian           234
Pizza             232
Mughlai           215
Ice Cream         178
Desserts          150
Burger            116
Biryani           112
Raw Meats         110
Beverages          79
Finger Food        78
Asian              77
Japanese           72
Mexican            62
European           62
Healthy Food       56
Indian             51
Seafood            49
Thai               29
Mediterranean      28
Lebanese           25
Breakfast          24
Brazilian          22
Tibetan            22
BBQ                18
Bengali            18
Tea                17
Kerala             14
Modern Indian      14
French             14
Kashmiri           13
Sushi              13
Steak              13
International      13
British            12
Juices           

### 2. Selecting Features for Cuisine Classification

In [9]:
df_classification.columns

Index(['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address',
       'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines',
       'Average Cost for two', 'Currency', 'Has Table booking',
       'Has Online delivery', 'Is delivering now', 'Switch to order menu',
       'Price range', 'Aggregate rating', 'Rating color', 'Rating text',
       'Votes', 'Cuisine_Target'],
      dtype='object')

### 2.1 Identifying the Target and Candidate Features

In [10]:
target = "Cuisine_Target"

candidate_features = [
    "Restaurant ID",
    "Restaurant Name",
    "Country Code",
    "City",
    "Address",
    "Locality",
    "Locality Verbose",
    "Longitude",
    "Latitude",
    "Average Cost for two",
    "Currency",
    "Has Table booking",
    "Has Online delivery",
    "Is delivering now",
    "Switch to order menu",
    "Price range",
    "Aggregate rating",
    "Rating color",
    "Rating text",
    "Votes"
]

print("Target:", target)
print("Candidate features:", candidate_features)

Target: Cuisine_Target
Candidate features: ['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address', 'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Average Cost for two', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu', 'Price range', 'Aggregate rating', 'Rating color', 'Rating text', 'Votes']


### 2.2 Inspecting Candidate Features

In [11]:
df_classification[candidate_features].info()

<class 'pandas.core.frame.DataFrame'>
Index: 9298 entries, 0 to 9550
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9298 non-null   int64  
 1   Restaurant Name       9298 non-null   object 
 2   Country Code          9298 non-null   int64  
 3   City                  9298 non-null   object 
 4   Address               9298 non-null   object 
 5   Locality              9298 non-null   object 
 6   Locality Verbose      9298 non-null   object 
 7   Longitude             9298 non-null   float64
 8   Latitude              9298 non-null   float64
 9   Average Cost for two  9298 non-null   int64  
 10  Currency              9298 non-null   object 
 11  Has Table booking     9298 non-null   object 
 12  Has Online delivery   9298 non-null   object 
 13  Is delivering now     9298 non-null   object 
 14  Switch to order menu  9298 non-null   object 
 15  Price range           9298

### 2.3 Inspecting Numerical Features

In [12]:
numerical_features = [
    "Longitude",
    "Latitude",
    "Average Cost for two",
    "Price range",
    "Aggregate rating",
    "Votes"
]

df_classification[numerical_features].describe()

,Longitude,Latitude,Average Cost for two,Price range,Aggregate rating,Votes
count,9298.000000,9298.000000,9298.000000,9298.000000,9298.000000,9298.000000
mean,65.023193,25.988966,1045.243063,1.786621,2.643214,151.746182
std,39.557607,10.535052,15066.113988,0.896653,1.515877,427.778597
min,-157.948486,-41.330428,0.000000,1.000000,0.000000,0.000000
25%,77.083652,28.480512,250.000000,1.000000,2.500000,4.000000
50%,77.193698,28.570916,400.000000,2.000000,3.200000,29.000000
75%,77.282708,28.642561,700.000000,2.000000,3.700000,124.000000
max,174.832089,55.976980,800000.000000,4.000000,4.900000,10934.000000


### 2.4 Final Feature Selection

The original Cuisines column is excluded because the classification target is derived from it.
The remaining relevant restaurant attributes are selected as input features for cuisine classification.

In [13]:
classification_features = [
    "Country Code",
    "City",
    "Longitude",
    "Latitude",
    "Average Cost for two",
    "Currency",
    "Has Table booking",
    "Has Online delivery",
    "Is delivering now",
    "Price range",
    "Aggregate rating",
    "Votes"
]

X = df_classification[classification_features]
y = df_classification["Cuisine_Target"]

print("Features:", classification_features)
print("Target:", y.name)

Features: ['Country Code', 'City', 'Longitude', 'Latitude', 'Average Cost for two', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Price range', 'Aggregate rating', 'Votes']
Target: Cuisine_Target


### 2.5 Identifying Categorical and Numerical Features

The selected input features are divided into categorical and numerical features so that appropriate preprocessing techniques can be applied before model training.

In [14]:
categorical_features = X.select_dtypes(include="object").columns.tolist()
numerical_features = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical Features:", categorical_features)
print("Numerical Features:", numerical_features)

Categorical Features: ['City', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now']
Numerical Features: ['Country Code', 'Longitude', 'Latitude', 'Average Cost for two', 'Price range', 'Aggregate rating', 'Votes']


### 2.6 Handling Country Code

Country Code represents country categories rather than a continuous numerical measurement.
Therefore, it is treated as a categorical feature for classification preprocessing.

In [16]:
X["Country Code"].value_counts()

Country Code
1      8539
216     374
215      69
30       55
214      52
189      52
148      33
208      23
166      18
184      17
191      17
162      16
14       15
94       14
37        4
Name: count, dtype: int64

### 2.7 Finalizing Feature Types

In [17]:
categorical_features = [
    "Country Code",
    "City",
    "Currency",
    "Has Table booking",
    "Has Online delivery",
    "Is delivering now"
]

numerical_features = [
    "Longitude",
    "Latitude",
    "Average Cost for two",
    "Price range",
    "Aggregate rating",
    "Votes"
]

print("Categorical Features:", categorical_features)
print("Numerical Features:", numerical_features)

Categorical Features: ['Country Code', 'City', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now']
Numerical Features: ['Longitude', 'Latitude', 'Average Cost for two', 'Price range', 'Aggregate rating', 'Votes']


## 3. Train-Test Split

The processed dataset is divided into training and testing sets so that the classification model can be trained on one portion of the data and evaluated on unseen data.

In [25]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train_raw.shape)
print("Testing data:", X_test_raw.shape)

Training data: (7438, 12)
Testing data: (1860, 12)


## 4. Feature Preprocessing

Categorical and numerical features are preprocessed using a pipeline.
Categorical features are encoded using One-Hot Encoding, while numerical features are standardized.
The preprocessing steps are fitted on the training data and then applied to the testing data to avoid data leakage.

In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print("Processed training data:", X_train.shape)
print("Processed testing data:", X_test.shape)

Processed training data: (7438, 158)
Processed testing data: (1860, 158)


## 5. Model Building

Classification models are developed using the preprocessed restaurant features to predict the primary cuisine category.

In [29]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

### 5.1 Training the Logistic Regression Model

In [31]:
logistic_model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


### 5.2 Making Predictions with Logistic Regression

The trained Logistic Regression model is used to predict the cuisine class for the restaurants in the testing dataset.

In [33]:
y_pred_logistic = logistic_model.predict(X_test)

y_pred_logistic[:20]

array(['North Indian', 'American', 'North Indian', 'North Indian',
       'North Indian', 'Cafe', 'North Indian', 'North Indian',
       'North Indian', 'North Indian', 'North Indian', 'North Indian',
       'American', 'North Indian', 'North Indian', 'Cafe', 'North Indian',
       'North Indian', 'North Indian', 'American'], dtype=object)

### 5.3 Evaluating Logistic Regression

In [34]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy_logistic = accuracy_score(y_test, y_pred_logistic)

precision_logistic = precision_score(
    y_test,
    y_pred_logistic,
    average="weighted",
    zero_division=0
)

recall_logistic = recall_score(
    y_test,
    y_pred_logistic,
    average="weighted",
    zero_division=0
)

print("Accuracy :", accuracy_logistic)
print("Precision:", precision_logistic)
print("Recall   :", recall_logistic)

Accuracy : 0.3424731182795699
Precision: 0.19536073071604104
Recall   : 0.3424731182795699


### 5.4 Detailed Logistic Regression Evaluation

In [35]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    y_test,
    y_pred_logistic,
    zero_division=0
))

               precision    recall  f1-score   support

     American       0.32      0.46      0.38        56
        Asian       0.00      0.00      0.00        15
          BBQ       0.00      0.00      0.00         4
       Bakery       0.11      0.01      0.02       124
      Bengali       0.00      0.00      0.00         4
    Beverages       0.00      0.00      0.00        16
      Biryani       0.00      0.00      0.00        22
    Brazilian       0.29      0.50      0.36         4
    Breakfast       0.00      0.00      0.00         5
      British       0.00      0.00      0.00         2
       Burger       0.00      0.00      0.00        23
         Cafe       0.26      0.08      0.12       123
      Chinese       0.33      0.01      0.01       171
  Continental       0.08      0.02      0.03        47
     Desserts       0.50      0.03      0.06        30
     European       0.00      0.00      0.00        12
    Fast Food       0.00      0.00      0.00       135
  Finger 

### 5.5 Random Forest Classifier

In [36]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

### 5.6 Training the Random Forest Model

In [37]:
random_forest_model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


### 5.7 Making Predictions with Random Forest

The trained Random Forest classifier is used to predict cuisine classes for the testing dataset.

In [38]:
y_pred_rf = random_forest_model.predict(X_test)

y_pred_rf[:20]

array(['North Indian', 'Burger', 'Cafe', 'North Indian', 'North Indian',
       'Cafe', 'North Indian', 'North Indian', 'North Indian', 'Chinese',
       'Ice Cream', 'North Indian', 'British', 'North Indian',
       'North Indian', 'Cafe', 'North Indian', 'North Indian',
       'North Indian', 'Mexican'], dtype=object)

### 5.8 Evaluating Random Forest

In [39]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy_rf = accuracy_score(y_test, y_pred_rf)

precision_rf = precision_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

recall_rf = recall_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

print("Accuracy :", accuracy_rf)
print("Precision:", precision_rf)
print("Recall   :", recall_rf)

Accuracy : 0.3161290322580645
Precision: 0.2599358974689884
Recall   : 0.3161290322580645


### 5.9 Detailed Random Forest Evaluation


In [40]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_rf,
    zero_division=0
))

               precision    recall  f1-score   support

     American       0.25      0.27      0.26        56
        Asian       0.11      0.07      0.08        15
          BBQ       0.00      0.00      0.00         4
       Bakery       0.23      0.22      0.23       124
      Bengali       0.00      0.00      0.00         4
    Beverages       0.00      0.00      0.00        16
      Biryani       0.11      0.05      0.06        22
    Brazilian       0.25      0.50      0.33         4
    Breakfast       0.00      0.00      0.00         5
      British       0.00      0.00      0.00         2
       Burger       0.13      0.09      0.11        23
         Cafe       0.36      0.35      0.35       123
      Chinese       0.19      0.13      0.15       171
  Continental       0.06      0.04      0.05        47
     Desserts       0.19      0.13      0.16        30
     European       0.00      0.00      0.00        12
    Fast Food       0.15      0.16      0.15       135
  Finger 

### 5.10 Decision Tree Classifier

In [49]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier(
    random_state=42
)

### 5.11 Training the Decision Tree Classifier

In [50]:
decision_tree_model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


### 5.12 Making Predictions with Decision Tree

In [52]:
y_pred_dt = decision_tree_model.predict(X_test)

y_pred_dt[:20]

array(['North Indian', 'Japanese', 'North Indian', 'North Indian',
       'Bakery', 'Cafe', 'Fast Food', 'North Indian', 'Juices',
       'North Indian', 'Ice Cream', 'Chinese', 'British', 'North Indian',
       'North Indian', 'Cafe', 'Italian', 'North Indian', 'North Indian',
       'Mexican'], dtype=object)

### 5.13 Evaluating the Decision Tree Classifier

In [53]:
accuracy_dt = accuracy_score(y_test, y_pred_dt)

precision_dt = precision_score(
    y_test,
    y_pred_dt,
    average="weighted",
    zero_division=0
)

recall_dt = recall_score(
    y_test,
    y_pred_dt,
    average="weighted",
    zero_division=0
)

print("Accuracy :", accuracy_dt)
print("Precision:", precision_dt)
print("Recall   :", recall_dt)

Accuracy : 0.23817204301075268
Precision: 0.24366321858894135
Recall   : 0.23817204301075268


### 5.14 Detailed Decision Tree Evaluation 

In [56]:
print(classification_report(
    y_test,
    y_pred_dt,
    zero_division=0
))

               precision    recall  f1-score   support

     American       0.19      0.18      0.18        56
        Asian       0.04      0.07      0.05        15
          BBQ       0.00      0.00      0.00         4
       Bakery       0.18      0.20      0.19       124
      Bengali       0.00      0.00      0.00         4
    Beverages       0.05      0.06      0.05        16
      Biryani       0.05      0.05      0.05        22
    Brazilian       0.40      0.50      0.44         4
    Breakfast       0.11      0.20      0.14         5
      British       0.00      0.00      0.00         2
       Burger       0.07      0.09      0.08        23
         Cafe       0.26      0.28      0.27       123
      Chinese       0.18      0.16      0.17       171
  Continental       0.12      0.13      0.12        47
     Desserts       0.10      0.13      0.12        30
     European       0.00      0.00      0.00        12
    Fast Food       0.19      0.19      0.19       135
  Finger 

## 6. Model Comparison

The performance of Logistic Regression, Random Forest, and Decision Tree classifiers is compared using accuracy, precision, and recall on the testing dataset.

In [55]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest","Decision Tree"],
    "Accuracy": [accuracy_logistic, accuracy_rf,accuracy_dt],
    "Precision": [precision_logistic, precision_rf,precision_dt],
    "Recall": [recall_logistic, recall_rf,recall_dt]
})

comparison

,Model,Accuracy,Precision,Recall
0,Logistic Regression,0.342473,0.195361,0.342473
1,Random Forest,0.316129,0.259936,0.316129
2,Decision Tree,0.238172,0.243663,0.238172


## 7. Cuisine-wise Performance Analysis

The performance of the classification models is analyzed across different cuisine categories to identify well-performing and poorly performing classes.

### 7.1 Challenges and Bias

The main challenges observed during cuisine classification are class imbalance, rare cuisine categories, similar restaurant characteristics across different cuisines, and the presence of restaurants associated with multiple cuisines.

Some cuisine categories achieved better performance than others, while several categories had very low or zero precision and recall. This indicates that the available restaurant features are not sufficient to reliably distinguish all cuisine categories.

### 7.2 Model Selection

Among the tested models, Logistic Regression achieved the highest overall accuracy of 34.25%. Random Forest achieved a weighted precision of 25.99%, while Decision Tree achieved an accuracy of 23.82%.

Based on overall accuracy, Logistic Regression is selected as the final model for this classification experiment.

## 8. Results and Interpretation
Three classification models were evaluated for cuisine prediction: Logistic Regression, Random Forest, and Decision Tree.

Logistic Regression achieved the highest overall accuracy of 34.25%. Random Forest achieved a weighted precision of 25.99%, while Decision Tree achieved an overall accuracy of 23.82%.

The results show that cuisine classification is challenging using the available restaurant features. The models performed better for some cuisines than others, while several cuisine categories had very low or zero precision and recall values.

## 9. Conclusion

A machine learning-based cuisine classification system was developed using the restaurant dataset. The data was explored, cuisine labels were prepared, rare cuisine categories were handled, and relevant restaurant features were selected.

Categorical features were encoded using One-Hot Encoding and numerical features were standardized. The dataset was divided into training and testing sets, and three classification models were evaluated: Logistic Regression, Random Forest, and Decision Tree.

Among the tested models, Logistic Regression achieved the highest overall accuracy of 34.25%. However, the cuisine-wise evaluation showed that several cuisine categories were difficult to classify accurately.

The main challenges observed were class imbalance, rare cuisine categories, similar restaurant characteristics across different cuisines, and restaurants associated with multiple cuisines. These findings highlight the limitations of the available features for reliable cuisine classification.